In [ ]:
import json

import pandas as pd
import plotly.express as px

In [ ]:
# (to export as html)
import plotly.offline as pyo

pyo.init_notebook_mode(connected=False)

In [ ]:
with open("../data/extra/measurements_991219.json") as f:
    data = json.load(f)

df = pd.DataFrame(data)[["pm1dot0", "pm2dot5", "pm10", "recorded"]]
df["recorded"] = pd.to_datetime(df["recorded"])
df = df.sort_values("recorded").reset_index(drop=True)

In [ ]:
df_m = df.melt(
    id_vars="recorded",
    value_vars=["pm1dot0", "pm10", "pm2dot5"],
    var_name="pm",
    value_name="reading"
)

In [ ]:
fig = px.line(
    df_m, 
    x='recorded', 
    y='reading', 
    color='pm', 
)

fig.show()

In [ ]:
df = df.set_index('recorded').sort_index()

In [ ]:
hourly_avg = df['pm2dot5'].resample('1h').mean().reset_index()

In [ ]:
px.line(hourly_avg, x="recorded", y="pm2dot5", markers=True)

The [
Technical Assistance Document for the Reporting of Daily Air Quality
](https://document.airnow.gov/technical-assistance-document-for-the-reporting-of-daily-air-quailty.pdf)
gives a very detailed explanation on how to calculate the AQI, using the values of:
- PM_2.5 (ug/m3)
- PM_10 (ug/m3)
- ozone (ppm)
- CO (ppm)
- SO_2 (ppb)
- NO_2 (ppb)


We only have the first two from this list.


For PM_10, it tells use to truncate to integer. For PM_2.5, to 1 decimal place.

The equation to map the pollutant values to the AQI is the following:

$$
I = \frac{I_{\text{high}} - I_{\text{low}}}{[BP]_{\text{high}} - [BP]_{\text{low}}}(C - [BP]_{\text{low}})+I_{\text{low}}
$$

where
- C: trucanted concentration of pollutant
- BP: concentrations breakpoints (low end and high end)
- I: index (low end and high end)


And the breakpoints table is

| PM2.5 [$\mu g / m^3 $]<br>$[BP]$|AQI<br>$I$|
|--|--|
|  0–12.0       |  0–50     |
|  12.1–35.4    |  51–100   |
|  35.5–55.4    |  101–150  |
|  55.5–150.4   |  151–200  |
|  150.5–250.4  |  201–300  |
|  250.5–350.4  |  301–400  |
|  350.5–500.4  |  401–500  |

This is the very same table used in the backend: https://github.com/tchx84/linka/blob/master/app/reports.py

(And only the PM2.5 value is actually used there)

In [ ]:
CONCENTRATIONS = [
    [0.0, 12.0],
    [12.1, 35.4],
    [35.5, 55.4],
    [55.5, 150.4],
    [150.5, 250.4],
    [250.5, 350.4],
    [350.5, 500.4],
]

BREAKPOINTS = [
    [0, 50],
    [51, 100],
    [101, 150],
    [151, 200],
    [201, 300],
    [301, 400],
    [401, 500],
]

So let's go back to the `hourly_average` dataframe and calculate the AQI.

In [ ]:
hourly_avg["c"] = hourly_avg["pm2dot5"].round(1)

In [ ]:
hourly_avg["concentration_breakpoints"] = hourly_avg.apply(
    lambda r: next(
        (c for c in CONCENTRATIONS if r["c"] <= c[1]),
        CONCENTRATIONS[-1],
    ), axis=1)

In [ ]:
hourly_avg["aqi_ranges"] = hourly_avg.apply(
    lambda r: BREAKPOINTS[CONCENTRATIONS.index(r["concentration_breakpoints"])],
    axis=1
)

In [ ]:
# lets get that slope first
def slope(concentrations, aqis):
    c_l, c_h = concentrations
    i_l, i_h = aqis

    return (i_h - i_l) / (c_h - c_l)


hourly_avg["slope"] = hourly_avg.apply(
    lambda r: slope(r["concentration_breakpoints"], r["aqi_ranges"]),
    axis=1
)

In [ ]:
hourly_avg["aqi"] = hourly_avg.apply(
    lambda r:
        r["slope"]*(r["c"] - r["concentration_breakpoints"][0])
        + r["aqi_ranges"][0],
    axis=1
).round(0).astype(int)

In [ ]:
hourly_avg[["recorded", "aqi"]]

**We have reproduced the previous AQI table!**

(this time  using the concentration data obtained from the backend and applying the formula ourselves)

Note: little differences seem to be caused by rounding to the nearest integer (here) vs rounding down (backend)